## 2.9 Preprocesamiento

### 2.9.1 Pipeline y alcance

La entrada original continúa siendo `close`. Se implementa un pipeline de preprocesamiento con `StandardScaler`, independiente para cada activo en la comprobación de esta sección. Cada instancia se ajusta exclusivamente con las observaciones elegibles del entrenamiento cronológico; la validación utiliza únicamente `transform`. No se reutilizan escaladores del EDA calculados sobre todo DEVELOPMENT.

El [Pipeline de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) agrupa las transformaciones que se ajustan. El parseo de fechas, la alineación horaria, la construcción determinista de la etiqueta y la selección de ventanas válidas se realizan antes, conservando la correspondencia entre X, y y fechas. Estas operaciones no estiman parámetros con validación. El escalado aprendido queda dentro del pipeline. Cuando se entrene el regresor, se integrará como etapa final y el pipeline completo se ajustará de nuevo en cada fold.

La ejecución actual verifica el preprocesamiento con una columna de cierre, sin entrenar OLS, SVR ni persistencia. Esta comprobación no fija la longitud definitiva de entrada de los modelos y no genera métricas de desempeño predictivo.

### 2.9.2 Faltantes y elegibilidad temporal

Los huecos horarios se mantienen en una rejilla de calendario. No se rellenan con ceros, medias, interpolaciones ni observaciones posteriores. No se ha establecido un mecanismo de ausencia que justifique una imputación específica; además, imputar precios alteraría los retornos y la volatilidad construida a partir de ellos.

Los cierres con final horario irregular se enmascaran para la construcción analítica, sin modificar los registros originales. Una entrada histórica requiere todos sus cierres válidos; una etiqueta requiere 25 cierres consecutivos para obtener los 24 retornos futuros. Se excluyen del conjunto supervisado las ventanas incompletas y se mantienen sus fechas para auditar la cobertura. Esto no elimina filas de los archivos fuente ni demuestra que la muestra elegible esté libre de sesgo de selección.

La exclusión de etiquetas inválidas afecta al entrenamiento y a la evaluación, no obliga a conocer el futuro para emitir una predicción: en operación, la elegibilidad de la entrada se determina solo con el histórico disponible. La etiqueta se comprueba cuando el horizonte se realiza. Los modelos se compararán sobre las mismas observaciones evaluables, con motivos de exclusión documentados.

### 2.9.3 Transformaciones y escalado

Se adopta como configuración inicial el cierre sin transformación logarítmica como entrada y su estandarización: a cada columna se le resta la media de entrenamiento y se divide por su escala de entrenamiento. [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) utiliza la desviación con divisor n y es sensible a valores extremos. La estandarización no vuelve normal ni estacionaria la serie y no elimina la multicolinealidad entre rezagos.

La transformación logarítmica se utiliza para construir los retornos del objetivo, no se aplica automáticamente a `close` como predictor. El objetivo conserva la desviación estándar centrada de 24 retornos, `ddof=0`, expresada en porcentaje y sin anualización. No se escala la etiqueta en esta comprobación. Una transformación posterior de entradas o del objetivo deberá evaluarse dentro de la validación temporal; cualquier transformación del objetivo exigirá volver a su escala común al calcular métricas.

La variación de niveles entre entrenamiento y validación observada en 2.6 puede producir valores estandarizados alejados de cero. No se centra de nuevo la validación para ocultar ese cambio ni se recortan sus valores a un rango arbitrario.

### 2.9.4 Codificación y tratamiento de valores extremos

No se requiere codificación categórica: `symbol` se utiliza para separar activos y las fechas para ordenar y dividir observaciones. No se incluyen como columnas predictoras. Los otros campos de mercado permanecen fuera del alcance acordado.

Los extremos válidos de `close` se conservan. Los hallazgos del EDA no justifican asumir que todo precio extremo sea un error. No se aplica eliminación por IQR ni winsorización y no se reutilizan umbrales estimados sobre todo DEVELOPMENT. Se exige que los precios utilizados sean finitos y positivos para construir retornos logarítmicos válidos. Si posteriormente se compara un escalado robusto o un tratamiento adicional, sus parámetros se ajustarán dentro de cada entrenamiento y su elección dependerá de validación interna.

### 2.9.5 Ingeniería temporal

La función `prepare_close` construye ventanas de L cierres: el último cierre observado y sus L−1 rezagos horarios. L debe especificarse explícitamente; no se selecciona en esta sección. Los desplazamientos se realizan sobre el calendario horario de cada activo, de modo que un hueco no se convierta artificialmente en una observación consecutiva. El ancla identifica la apertura de la última vela utilizada y la predicción se sitúa después de su cierre.

La comprobación ejecutada utiliza L=1 únicamente para verificar el pipeline básico. Si se amplía L, cada rezago será una columna y se ajustará su escala con el entrenamiento. La fuente seguirá siendo `close`, aunque puedan aparecer dependencias fuertes entre columnas. No se añaden automáticamente retornos, volatilidad pasada, medias móviles, senos, cosenos ni indicadores de calendario como entradas. La estacionalidad semanal débil de 2.6 no aporta por sí sola evidencia suficiente para incorporarlos. No existen variables espaciales aplicables.

### 2.9.6 Decisiones vinculadas al EDA

| Hallazgo | Decisión de preprocesamiento |
|---|---|
| Huecos y cierres irregulares | Mantener el calendario y excluir ventanas incompletas; no imputar. |
| Diferencias de nivel y escala entre activos y periodos | Verificar escalado por activo, ajustado solo en entrenamiento. |
| Precios extremos que pueden ser movimientos reales | Conservar extremos válidos, sin recorte automático. |
| Persistencia y redundancia entre rezagos | Mantener L explícito; comparar ventanas dentro de validación, sin selección definitiva aquí. |
| Componente semanal débil en el tramo estudiado | No incorporar características de calendario automáticamente. |
| Ausencia de predictores categóricos y coordenadas | No usar codificación categórica ni ingeniería espacial. |
| Etiquetas que abarcan 24 horas futuras | Purgar por la fecha final de la etiqueta, antes de ajustar el pipeline. |

### 2.9.7 Verificación ejecutada

Se utiliza la frontera interna **2024-07-09 20:00 UTC**, dentro de DEVELOPMENT, como en la auditoría 2.5. Se excluyen del entrenamiento las etiquetas cuya última vela objetivo alcanza o supera esa frontera. Se ajustan cinco pipelines de escalado, uno por activo, con una sola columna de cierre. Las cifras siguientes son parámetros y tamaños de muestra, no resultados de un modelo predictivo.

| Activo | Filas entrenamiento | Filas validación | Etiquetas purgadas | Media entrenamiento (USDT) | Escala entrenamiento (USDT) |
|---|---:|---:|---:|---:|---:|
| BNBUSDT | 33972 | 8543 | 24 | 316.821586 | 159.145884 |
| BTCUSDT | 33972 | 8543 | 24 | 36050.904935 | 16026.223094 |
| ETHUSDT | 33972 | 8543 | 24 | 2140.664848 | 1013.130489 |
| SOLUSDT | 33997 | 8543 | 24 | 61.749436 | 60.355696 |
| XRPUSDT | 33997 | 8543 | 24 | 0.587047 | 0.271218 |

Las comprobaciones confirman que la media aprendida corresponde al entrenamiento, que transformar validación no cambia los parámetros, que las salidas son finitas y que la transformación inversa recupera los cierres de validación. La media estandarizada de entrenamiento es aproximadamente cero; no se exige que la de validación lo sea. Esta verificación de una frontera no sustituye las comprobaciones de los folds definitivos.

### 2.9.8 Reproducibilidad

El procedimiento está en `src/17_preprocessing_close.py`. La auditoría se guarda en `outputs/tables/preprocessing_close_audit.csv` y la trazabilidad en `outputs/tables/preprocessing_metadata.json`. El script verifica el SHA-256 de DEVELOPMENT antes y después de ejecutarse. No se modifica ningún dataset ni se lee TEST. Los pipelines de esta comprobación no se guardan como modelos finales ni se reutilizan para otros folds.

In [1]:
"""Pipeline de escalado y verificación temporal, sin estimador predictivo."""
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/splits/development_80.csv').is_file())
def make_preprocessor():
    return Pipeline([('scale',StandardScaler())])
def prepare_close(g,grid,window):
    if not isinstance(window,int) or window<1:raise ValueError('window debe ser entero positivo')
    g=g.set_index('open_time').reindex(grid)
    valid=g.close_time.eq(grid+pd.Timedelta(hours=1)-pd.Timedelta(milliseconds=1))
    c=g.close.where(valid & np.isfinite(g.close) & g.close.gt(0))
    X=pd.DataFrame({f'close_lag_{lag}h':c.shift(lag) for lag in range(window)},index=grid)
    r=np.log(c).diff();y=100*r.rolling(24,min_periods=24).std(ddof=0).shift(-24)
    eligible=X.notna().all(axis=1)&y.notna()
    return X.loc[eligible],y.loc[eligible]
def main():
    source=ROOT/'data/splits/development_80.csv';sha=hashlib.sha256(source.read_bytes()).hexdigest()
    df=pd.read_csv(source,usecols=['symbol','open_time','close_time','close'])
    for field in ['open_time','close_time']:df[field]=pd.to_datetime(df[field],utc=True,format='ISO8601')
    assert not df.duplicated(['symbol','open_time']).any()
    times=pd.DatetimeIndex(sorted(df.open_time.unique()));cut=times[int(.8*len(times))]
    grid=pd.date_range(times.min(),times.max(),freq='h');rows=[]
    for symbol,g in df.groupby('symbol'):
        X,y=prepare_close(g,grid,window=1)
        train=X.index+pd.Timedelta(hours=24)<cut;val=X.index>=cut
        assert X.index[train].max()+pd.Timedelta(hours=24)<X.index[val].min()
        pipe=make_preprocessor();ztrain=pipe.fit_transform(X.loc[train])
        scaler=pipe.named_steps['scale'];mu=scaler.mean_.copy();var=scaler.var_.copy()
        zval=pipe.transform(X.loc[val])
        assert np.isfinite(ztrain).all() and np.isfinite(zval).all()
        assert np.allclose(mu,X.loc[train].mean().to_numpy())
        assert np.array_equal(mu,scaler.mean_) and np.array_equal(var,scaler.var_)
        assert np.allclose(ztrain.mean(axis=0),0,atol=1e-10)
        assert np.allclose(pipe.inverse_transform(zval),X.loc[val])
        assert np.allclose(pipe.transform(X.loc[val]*2), (X.loc[val]*2-mu)/scaler.scale_)
        rows.append(dict(symbol=symbol,train_rows=int(train.sum()),validation_rows=int(val.sum()),purged_rows=int((~train&~val).sum()),training_mean=float(mu[0]),training_scale=float(scaler.scale_[0]),train_z_mean=float(ztrain.mean()),validation_z_mean=float(zval.mean())))
    assert sha==hashlib.sha256(source.read_bytes()).hexdigest()
    out=ROOT/'outputs/tables';result=pd.DataFrame(rows);result.to_csv(out/'preprocessing_close_audit.csv',index=False)
    (out/'preprocessing_metadata.json').write_text(json.dumps({'source':'data/splits/development_80.csv','sha256':sha,'validation_anchor_start':str(cut),'demonstration_window':1,'final_window_selected':False,'imputation':False,'outliers_removed':False,'predictive_model_trained':False,'test_read':False,'sklearn':sklearn.__version__},indent=2),encoding='utf-8')
    print(result.to_string(index=False))
if __name__=='__main__':main()


 symbol  train_rows  validation_rows  purged_rows  training_mean  training_scale  train_z_mean  validation_z_mean
BNBUSDT       33972             8543           24     316.821586      159.145884  2.944908e-16           1.875061
BTCUSDT       33972             8543           24   36050.904935    16026.223094  3.078768e-16           3.029602
ETHUSDT       33972             8543           24    2140.664848     1013.130489 -1.405524e-16           0.533104
SOLUSDT       33997             8543           24      61.749436       60.355696  2.675220e-17           1.770291
XRPUSDT       33997             8543           24       0.587047        0.271218  2.173617e-17           3.994497
